# Ice Dynamics I: Exercises

**Ge 193 — Glacier Dynamics, Spring 2026**

These exercises accompany the lecture *Ice Dynamics I: Strain Rates, Simple Shear, and Ice Sheet Profiles*. They are designed to build intuition for the parallel-slab velocity profile and the Vialov ice-sheet profile through computation and visualization.

Each exercise provides scaffolding code. Look for `# TODO` comments indicating where you need to fill in expressions.

In [ ]:
# ---- Setup: imports and physical constants ----
import numpy as np
import matplotlib.pyplot as plt

rho_i = 917.0       # kg/m^3, ice density
g = 9.81            # m/s^2
sec_per_yr = 365.25 * 24 * 3600  # seconds per year

n = 3               # creep exponent
A = 2.4e-24         # Pa^{-n} s^{-1}, rate factor (approx. -10 C)

print('Setup complete.')

---
## Exercise 1: Velocity Profiles for the Parallel Slab

The velocity profile in a parallel slab of ice with no basal sliding is:

$$u(z) = \frac{2A}{n+1}(\rho_i g \sin\alpha)^n \left[h^{n+1} - (s - z)^{n+1}\right]$$

where $z$ is the height above some reference, $s = b + h$ is the surface elevation, and $b$ is the bed elevation.

**Tasks:**
1. Write a function that computes $u(z)$ given the parameters below.
2. Plot the velocity profile for $n = 1, 3, 5$ on the same axes.
3. For each $n$, find the height at which $u = 0.5\,u_s$ (half the surface velocity). What does this tell you about where deformation is concentrated?

In [ ]:
# ---- Ex1: slab velocity profile u(z) for different n ----
h = 300.0           # m, ice thickness
alpha_deg = 5.0     # degrees, surface slope
alpha = np.radians(alpha_deg)
z = np.linspace(0, h, 500)


def velocity_profile(z, h, alpha, A, n):
    """Compute the velocity at height z above the bed for a parallel slab."""
    # TODO: implement the velocity profile formula
    # Hint: s - z = h - z when the bed is at z = 0
    u = np.zeros_like(z)  # replace this line
    return u/max(u)


# Task 2: plot profiles for different n
fig, ax = plt.subplots(figsize=(5, 6))

for n_val in [1, 3, 5]:
    u = velocity_profile(z, h, alpha, A, n_val)
    u_s = u[-1]  # surface velocity
    # TODO: convert u to m/yr and plot
    # ax.plot(..., label=f'n = {n_val}')

ax.set_xlabel('Velocity (m/yr)')
ax.set_ylabel('Height above bed (m)')
ax.legend()
ax.set_title('Parallel slab velocity profiles')
plt.show()

# Task 3: half-velocity height
# From u/u_s = 1 - (1 - z/h)^{n+1}, setting u/u_s = 0.5 gives z/h = 1 - 0.5^{1/(n+1)}
print('\nHeight where u = 0.5 * u_s (as fraction of h):')
for n_val in [1, 3, 5]:
    z_half = 0  # TODO: replace with the correct expression
    print(f'  n = {n_val}: z/h = {z_half:.3f}')

---
## Exercise 2: Numerical Integration of the Velocity Profile

The depth-averaged velocity for the parallel slab (no sliding) is:

$$\bar{u} = \frac{n+1}{n+2}\,u_s$$

As $n$ increases, ice becomes more nonlinear and the velocity profile becomes more plug-like. In the limit $n \to \infty$, all deformation is confined to a thin basal layer and $\bar{u}/u_s \to 1$.

**Tasks:**
1. For each integer $n$ from 1 to 10, use `np.trapz` to numerically compute $\bar{u}/u_s$ from the normalized velocity profile $u/u_s = 1 - (1 - \hat{z})^{n+1}$.
2. Plot $\bar{u}/u_s$ vs. $n$, showing both the numerical values (markers) and the analytical curve $(n+1)/(n+2)$. Add a horizontal dashed line at $\bar{u}/u_s = 1$ to indicate the plug-flow limit.
3. At what value of $n$ does the ratio exceed 0.9? What does this tell you about how quickly the flow approaches plug-like behavior?

In [ ]:
# ---- Ex2: plot ubar/us vs n showing convergence to plug flow ----
n_values = np.arange(1, 11)  # n = 1, 2, ..., 10
zhat = np.linspace(0, 1, 500)  # normalized height (z - b) / h

ratio_numerical = np.zeros(len(n_values))
ratio_analytical = np.zeros(len(n_values))

for i, n_val in enumerate(n_values):
    u_norm = 1 - (1 - zhat)**(n_val + 1)
    
    # TODO: u_bar/u_s = integral of (u/u_s) d(zhat) from 0 to 1
    ratio_numerical[i] = 0  # replace this
    
    ratio_analytical[i] = (n_val + 1) / (n_val + 2)

fig, ax = plt.subplots(figsize=(7, 4.5))

n_smooth = np.linspace(1, 10, 200)
ratio_smooth = (n_smooth + 1) / (n_smooth + 2)
ax.plot(n_smooth, ratio_smooth, '-', color='#1f77b4', lw=2, label='Analytical: $(n+1)/(n+2)$')

# TODO: plot ratio_numerical vs n_values as markers
# ax.plot(n_values, ratio_numerical, ...)

ax.axhline(y=1.0, color='k', ls='--', lw=1, alpha=0.5, label='Plug-flow limit ($n \\to \\infty$)')

ax.set_xlabel('Creep exponent $n$')
ax.set_ylabel('$\\bar{u} \\, / \\, u_s$')
ax.set_xlim(0.5, 10.5)
ax.set_ylim(0.6, 1.05)
ax.legend(frameon=False)
ax.set_title('Convergence to plug flow with increasing $n$')
plt.show()

---
## Exercise 3: Vialov Profile — Parameter Sensitivity

The Vialov profile for a symmetric ice sheet on a flat bed with uniform accumulation is:

$$h(x) = h_0 \left[1 - \left(\frac{x}{L}\right)^{(n+1)/n}\right]^{n/(2(n+1))}$$

with divide thickness

$$h_0 = \left[\frac{2^{n-1}(n+2)\,\dot{b}_i}{A\,(\rho_i g)^n}\right]^{1/(2n+2)} L^{(n+1)/(2n+2)}$$

**Tasks:**
1. Write functions for $h_0$ and $h(x)$.
2. Make a 2\times2 panel figure varying one parameter at a time ($\dot{b}_i$, $A$, $L$, $n$) while holding the others fixed. Use the default values below.
3. For each parameter, compute the *ratio* $h_0^{\max}/h_0^{\min}$ across the range you explored. Which parameter has the strongest influence? Which has the weakest? Does this match the analytical scaling?

In [ ]:
# ---- Ex3: Vialov profile 2x2 parameter sensitivity ----
n_default = 3
A_default = 2.4e-24
L_default = 1000e3
bdot_default = 0.15 / sec_per_yr


def vialov_h0(bdot_i, A, L, n):
    """Compute the divide thickness for the Vialov profile."""
    # TODO: implement the formula for h0
    h0 = 0  # replace this
    return h0


def vialov_profile(x, L, h0, n):
    """Compute the Vialov profile h(x)."""
    # TODO: implement the Vialov profile formula
    h = np.zeros_like(x)  # replace this
    return h


fig, axes = plt.subplots(2, 2, figsize=(11, 8))

# Panel (a): vary bdot_i
ax = axes[0, 0]
ax.set_title('(a) Varying accumulation rate')
bdot_values = [0.05, 0.10, 0.15, 0.30, 0.50]  # m/yr
x = np.linspace(0, L_default, 1000)
for bdot_yr in bdot_values:
    bdot = bdot_yr / sec_per_yr
    h0 = vialov_h0(bdot, A_default, L_default, n_default)
    h = vialov_profile(x, L_default, h0, n_default)
    ax.plot(x / 1e3, h, label=f'{bdot_yr} m/yr')
ax.set_xlabel('Distance from divide (km)')
ax.set_ylabel('Thickness (m)')
ax.legend(fontsize=9)

# Panel (b): vary A
ax = axes[0, 1]
ax.set_title('(b) Varying rate factor $A$')
A_values = [3.5e-26, 2.4e-25, 2.4e-24, 2.4e-23]
A_labels = ['3.5e-26', '2.4e-25', '2.4e-24', '2.4e-23']
for A_val, A_lab in zip(A_values, A_labels):
    h0 = vialov_h0(bdot_default, A_val, L_default, n_default)
    h = vialov_profile(x, L_default, h0, n_default)
    ax.plot(x / 1e3, h, label=f'$A$ = {A_lab}')
ax.set_xlabel('Distance from divide (km)')
ax.set_ylabel('Thickness (m)')
ax.legend(fontsize=9)

# Panel (c): vary L
ax = axes[1, 0]
ax.set_title('(c) Varying half-span $L$')
L_values = [200e3, 500e3, 1000e3, 2000e3]
for L_val in L_values:
    x_L = np.linspace(0, L_val, 1000)
    h0 = vialov_h0(bdot_default, A_default, L_val, n_default)
    h = vialov_profile(x_L, L_val, h0, n_default)
    ax.plot(x_L / 1e3, h, label=f'$L$ = {L_val/1e3:.0f} km')
ax.set_xlabel('Distance from divide (km)')
ax.set_ylabel('Thickness (m)')
ax.legend(fontsize=9)

# Panel (d): vary n
ax = axes[1, 1]
ax.set_title('(d) Varying creep exponent $n$')
n_values = [1, 2, 3, 5]
for n_val in n_values:
    h0 = vialov_h0(bdot_default, A_default, L_default, n_val)
    h = vialov_profile(x, L_default, h0, n_val)
    ax.plot(x / 1e3, h, label=f'$n$ = {n_val}')
ax.set_xlabel('Distance from divide (km)')
ax.set_ylabel('Thickness (m)')
ax.legend(fontsize=9)

fig.tight_layout()
plt.show()

# Task 3: compute h0_max / h0_min for each parameter variation
# TODO: print a table of ratios and compare with the analytical scaling exponents

---
## Exercise 4: Perfectly Plastic Profile and the $n \to \infty$ Limit

The perfectly plastic profile on a flat bed is:

$$h^2(x) = \frac{2\tau_0}{\rho_i g}(L - x)$$

**Tasks:**
1. Plot the plastic profile and the Vialov profile ($n=3$) on the same axes using dimensional values ($L = 1000$ km, $\tau_0 = 100$ kPa). Choose $\dot{b}_i$ and $A$ so that the Vialov divide thickness is similar to the plastic one (or just normalize both to $h_0 = 1$).
2. Compute the Vialov profile for $n = 1, 3, 5, 10, 20, 50$ and overlay them with the plastic profile. At what $n$ does the Vialov profile become visually indistinguishable from the plastic one?
3. **Bonus:** The surface slope $|dh/dx|$ diverges at the margin for both profiles. Compute and plot $|dh/dx|$ as a function of $x/L$ for both. Where does the SIA assumption of small surface slopes break down?

In [ ]:
# ---- Ex4: plastic vs Vialov profiles, convergence as n->inf ----
tau_0 = 100e3  # Pa, yield stress
L = 1000e3     # m

x_norm = np.linspace(0, 1, 1000)

h_plastic_norm = np.sqrt(np.maximum(1 - x_norm, 0))

# Task 1: plot plastic and Vialov (n=3) profiles
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(x_norm, h_plastic_norm, '--', lw=2.5, label='Perfectly plastic')

# TODO: compute and plot the normalized Vialov profile for n = 3
# Hint: h/h0 = [1 - (x/L)^{(n+1)/n}]^{n/(2(n+1))}

ax.set_xlabel('$x / L$')
ax.set_ylabel('$h / h_0$')
ax.legend()
ax.set_title('Plastic vs. Vialov profiles')
plt.show()

# Task 2: convergence to plastic limit
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(x_norm, h_plastic_norm, 'k--', lw=3, label='Plastic ($n \\to \\infty$)')

for n_val in [1, 3, 5, 10, 20, 50]:
    # TODO: compute and plot the Vialov profile for this n
    pass

ax.set_xlabel('$x / L$')
ax.set_ylabel('$h / h_0$')
ax.legend(fontsize=9)
ax.set_title('Vialov profiles converging to the plastic limit')
plt.show()

---
## Exercise 5: Balance Velocity from the Vialov Profile

In the mass balance lecture, we defined the balance velocity as the depth-averaged velocity needed to maintain steady state:

$$U_B(x) = \frac{q(x)}{h(x)} = \frac{\dot{b}_i\, x}{h(x)}$$

for the 2D ice sheet with uniform accumulation (since $q = \dot{b}_i x$ at steady state).

The SIA also gives us the depth-averaged velocity from the deformational flux:

$$\bar{u}(x) = \frac{2A}{n+2}(\rho_i g)^n \left|\frac{dh}{dx}\right|^n h^{n+1}$$

Since the Vialov profile is the steady-state solution, these two expressions *must* agree.

**Tasks:**
1. Compute $U_B(x) = \dot{b}_i x / h(x)$ from the Vialov thickness.
2. Compute $\bar{u}(x)$ from the SIA flux using a numerical derivative of $h(x)$ (use `np.gradient`).
3. Plot both on the same axes and verify they match. Where does the numerical derivative become inaccurate, and why?
4. What happens to the velocity at the margin ($x \to L$)? Is this physically reasonable?

In [ ]:
# ---- Ex5: balance velocity vs SIA velocity from Vialov profile ----
n = 3
L = 1000e3
bdot = 0.15 / sec_per_yr  # m/s

x = np.linspace(0, L * 0.99, 1000)  # avoid the margin singularity

h0 = vialov_h0(bdot, A, L, n)
h = vialov_profile(x, L, h0, n)

# Task 1: balance velocity from continuity
# TODO: U_B = bdot * x / h (be careful about x = 0)
U_B = np.zeros_like(x)

# Task 2: depth-averaged velocity from SIA flux
dhdx = np.gradient(h, x)

# TODO: compute u_bar from the SIA formula
u_bar = np.zeros_like(x)

# Task 3: plot both
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(x / 1e3, U_B * sec_per_yr, '-', lw=2, label='$U_B = \\dot{b}_i x / h$')
ax.plot(x / 1e3, u_bar * sec_per_yr, '--', lw=2, label='$\\bar{u}$ from SIA flux')

ax.set_xlabel('Distance from divide (km)')
ax.set_ylabel('Velocity (m/yr)')
ax.legend()
ax.set_title('Balance velocity vs. SIA velocity')
plt.show()